# Le pape François adapte-t-il ses discours aux pays qu'il visite ?

Projet de Python pour la data science, ENSAE, 2026. Auteurs : Quentin Brisemur, Arthur De Rouck, Adrien Vibert.

Entre 2013 et 2024, le pape François a fait 47 voyages hors d'Italie. Dans chaque pays, il prononce plusieurs discours devant des publics très différents : chefs d'État, évêques, jeunes, réfugiés, responsables d'autres religions. Le Vatican publie ces textes en anglais sur son site.

On cherche à savoir si ces discours changent selon le lieu et le public, ou s'ils reprennent partout les mêmes thèmes. Pour cela, on a collecté tous les discours prononcés pendant ces voyages, puis on les compare par continent, par type de public et dans le temps.

Le rapport suit quatre étapes : les données, la description du corpus, l'analyse, puis les limites. Les graphiques sont interactifs : on peut les survoler pour lire les valeurs.

> À rédiger. Quelques lignes sur l'intérêt de la question et sur ce que vous attendiez avant de regarder les données.

In [1]:
import sys
import textwrap
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RACINE / "outils"))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import analyse_corpus as ac
import graphiques as g

g.style()
pd.set_option("display.max_colwidth", 70)

## 1. Les données

### D'où viennent les discours

Les textes viennent du site vatican.va. Deux scripts du dossier `acquisition` produisent les données.

1. `telecharger_discours.py` lit la liste des voyages hors d'Italie et leurs dates. Il parcourt ensuite les index mensuels des discours et garde ceux prononcés pendant un voyage. On obtient 415 textes.
2. `preparer_corpus.py` sépare l'en-tête de chaque texte (voyage, public, lieu, date) du discours lui-même. Il retire les conférences de presse dans l'avion, qui contiennent surtout les questions des journalistes, et deux discours prononcés au Vatican le matin d'un départ. Il reste 342 discours.

Les fichiers produits sont déposés sur le stockage du SSP Cloud. Le notebook les lit directement, sans relancer la collecte qui prend une dizaine de minutes.

In [2]:
df = ac.charger()
df[["date", "voyage", "pays", "public", "n_mots", "titre"]].head()

,date,voyage,pays,public,n_mots,titre
0,2013-07-22,Rio de Janeiro 2013,Brésil,Accueil et départ,990,Welcome Ceremony at the Garden of Guanabara Palace (Rio de Janeiro...
1,2013-07-24,Rio de Janeiro 2013,Brésil,Jeunes et éducation,111,Greeting to the young people from Italy at the conclusion of his v...
2,2013-07-24,Rio de Janeiro 2013,Brésil,"Pauvres, malades, migrants",999,Visit to the St. Francis of Assisi of the Providence of God Hospit...
3,2013-07-24,Rio de Janeiro 2013,Brésil,"Prière, mémoire, réconciliation",168,Impromptu address of Pope Francis from the balcony of the Basilica...
4,2013-07-25,Rio de Janeiro 2013,Brésil,Jeunes et éducation,2155,Welcoming Ceremony by the young people on the waterfront of Copaca...


### Variables construites

La table contient une ligne par discours. La fonction `ac.charger()` ajoute trois informations.

- Le pays et le continent. Le lieu écrit dans l'en-tête est comparé à une liste de pays et de villes. S'il manque, on prend le pays du voyage quand il n'y en a qu'un, sinon celui des autres discours du même jour.
- Le type de public, déduit des mots du titre (« Meeting with Authorities », « Meeting with Young People »...).
- La longueur du discours en nombre de mots.

Le tableau suivant résume le corpus et vérifie qu'aucun discours n'est resté sans pays.

In [3]:
pd.Series({
    "Discours": len(df),
    "Voyages": df["voyage"].nunique(),
    "Pays": df["pays"].nunique(),
    "Discours sans pays attribué": df["pays"].isna().sum(),
    "Discours de moins de 150 mots": (df["n_mots"] < 150).sum(),
    "Mots au total": df["n_mots"].sum(),
}, name="Nombre").to_frame()

,Nombre
Discours,342
Voyages,47
Pays,66
Discours sans pays attribué,0
Discours de moins de 150 mots,8
Mots au total,543217


## 2. Description du corpus

### Où est-il allé ?

In [4]:
par_pays = (df.groupby(["iso", "pays"])
              .agg(discours=("titre", "size"), voyages=("voyage", "nunique"))
              .reset_index())

fig = px.choropleth(par_pays, locations="iso", color="discours", hover_name="pays",
                    custom_data=["voyages"], color_continuous_scale=g.BLEUS[2:])
fig.update_traces(marker_line_color="white", marker_line_width=0.5,
                  hovertemplate="%{hovertext}<br>%{z} discours, %{customdata[0]} voyage(s)<extra></extra>")
fig.update_geos(projection_type="natural earth", showframe=False, showcoastlines=False, lataxis_range=[-58, 85],
                showcountries=True, countrycolor="white", landcolor="#ecebe6", bgcolor=g.FOND)
fig.update_layout(coloraxis_colorbar=dict(title="Discours", thickness=12, len=0.6), margin=dict(l=0, r=0, b=0))
g.titre(fig, "Nombre de discours prononcés dans chaque pays", "Survolez un pays pour voir le détail", hauteur=500)
fig.show()

> À rédiger. Quels pays et quelles régions reviennent le plus ? Quelles grandes régions sont absentes ?

### Quand ?

In [5]:
par_annee = df.groupby(["annee", "continent"]).size().rename("discours").reset_index()
totaux = df.groupby("annee").size().reindex(range(2013, 2025), fill_value=0)

fig = px.bar(par_annee, x="annee", y="discours", color="continent",
             category_orders={"continent": ac.ORDRE_CONTINENTS}, color_discrete_map=g.COULEUR_CONTINENT)
fig.update_traces(marker_line_color=g.FOND, marker_line_width=1.5,
                  hovertemplate="%{x}, %{fullData.name} : %{y} discours<extra></extra>")
fig.add_scatter(x=totaux.index, y=totaux + 3, text=totaux.where(totaux > 0, ""), mode="text",
                showlegend=False, hoverinfo="skip", textfont=dict(color=g.ENCRE_2))
fig.add_annotation(x=2020, y=10, text="aucun voyage", textangle=-90, showarrow=False, font=dict(size=12, color=g.GRIS))
fig.update_xaxes(title=None, dtick=1)
fig.update_yaxes(title="Discours", showgrid=True)
g.titre(fig, "Discours prononcés chaque année, selon le continent",
        "Cliquez sur un continent dans la légende pour le masquer", hauteur=500)
fig.show()

> À rédiger. Comment le rythme des voyages évolue-t-il ? Que se passe-t-il en 2020 et après ? Les continents se succèdent-ils ou se mélangent-ils ?

### À qui parle-t-il ?

Le type de public est déduit du titre. Par exemple « Meeting with the Authorities, Civil Society and the Diplomatic Corps » est classé dans « Autorités et diplomatie ».

In [6]:
publics = df["public"].value_counts().sort_values()

fig = go.Figure(go.Bar(x=publics.values, y=publics.index, orientation="h", marker_color=g.COULEURS[0],
                       text=publics.values, textposition="outside", cliponaxis=False,
                       hovertemplate="%{y} : %{x} discours<extra></extra>"))
fig.update_xaxes(showgrid=True)
g.titre(fig, "Nombre de discours par type de public", hauteur=460)
fig.show()

> À rédiger. Quels publics dominent ? Qu'est-ce que cela dit de la nature de ces voyages ?

### Combien de temps parle-t-il ?

In [7]:
ordre = df.groupby("public")["n_mots"].median().sort_values().index.tolist()
medianes = df.groupby("public")["n_mots"].median().reindex(ordre)

fig = px.strip(df, x="n_mots", y="public", hover_name="titre", custom_data=["voyage"],
               category_orders={"public": ordre[::-1]})
fig.update_traces(marker=dict(color=g.COULEURS[0], opacity=0.45, size=7), jitter=0.7,
                  hovertemplate="%{hovertext}<br>%{customdata[0]}, %{x} mots<extra></extra>")
fig.add_scatter(x=medianes.values, y=medianes.index, mode="markers", showlegend=False,
                marker=dict(symbol="line-ns-open", size=26, color=g.ENCRE, line=dict(width=3, color=g.ENCRE)),
                hovertemplate="Médiane : %{x:.0f} mots<extra></extra>")
fig.update_xaxes(title="Nombre de mots", showgrid=True)
fig.update_yaxes(title=None)
g.titre(fig, "Longueur des discours selon le public",
        "Un point par discours, le trait noir marque la médiane. Survolez un point pour lire son titre.", hauteur=540)
fig.show()

> À rédiger. Quels discours sont les plus longs, lesquels les plus courts ? Est-ce attendu ?

## 3. Le discours change-t-il selon le pays et le public ?

On mesure l'adaptation de trois façons : la mention du pays hôte, les thèmes abordés et les mots propres à chaque continent. On termine par l'évolution dans le temps. L'Océanie, qui ne compte que 5 discours, est laissée de côté dans les comparaisons par continent, comme les publics de moins de 10 discours.

### Parle-t-il du pays où il se trouve ?

On regarde si le texte nomme le pays où le discours est prononcé, par exemple « Kenya » dans un discours à Nairobi.

In [8]:
df["cite_hote"] = ac.cite_le_pays_hote(df)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Par continent", "Par public"], horizontal_spacing=0.3)
for colonne, groupe in enumerate(["continent", "public"], start=1):
    parts = df.groupby(groupe)["cite_hote"].agg(["mean", "size"])
    parts = parts[parts["size"] >= 10].sort_values("mean")
    fig.add_bar(x=parts["mean"], y=parts.index, orientation="h", row=1, col=colonne, showlegend=False,
                width=0.5 if groupe == "continent" else 0.65,
                marker_color=[g.COULEUR_CONTINENT.get(nom, g.COULEURS[0]) for nom in parts.index],
                text=[f"{p:.0%}" for p in parts["mean"]], textposition="outside", cliponaxis=False,
                customdata=parts["size"], hovertemplate="%{y}<br>%{x:.0%} des %{customdata} discours<extra></extra>")
fig.update_xaxes(range=[0, 1.1], tickformat=".0%", showgrid=True)
g.titre(fig, "Part des discours qui nomment le pays hôte", "Groupes d'au moins 10 discours", hauteur=480)
fig.show()

> À rédiger. Le pays hôte est-il presque toujours nommé ? Pour quels continents et quels publics l'est-il moins ? Comment l'expliquer ?

### De quoi parle-t-il selon le continent ?

On définit 15 thèmes par des listes de mots anglais. Par exemple, le thème « Paix et guerre » regroupe « peace », « war », « weapons », « conflict »... Pour chaque continent, on compte les mentions de chaque thème pour 10 000 mots. La mesure est simple et vérifiable, mais elle compte des mots et non des idées.

La couleur se lit ligne par ligne : la case la plus foncée indique le continent où le thème revient le plus, la plus claire celui où il revient le moins. Les listes de mots sont dans `outils/analyse_corpus.py`.

In [9]:
grands_continents = [c for c in ac.ORDRE_CONTINENTS if (df["continent"] == c).sum() >= 10]
taux = ac.taux_lexique(df, "continent").loc[grands_continents].T
taux = taux.loc[taux.mean(axis=1).sort_values(ascending=False).index]

g.carte_chaleur(taux, "Mentions de chaque thème pour 10 000 mots, par continent",
                "Chaque ligne va du plus clair (minimum) au plus foncé (maximum)").show()

> À rédiger. Quels thèmes varient le plus d'un continent à l'autre ? Lesquels restent stables partout ? Les écarts correspondent-ils à ce qu'on sait des pays visités ?

### Et selon le public ?

Même mesure, en comparant les types de public qui comptent au moins 20 discours.

In [10]:
grands_publics = df["public"].value_counts().loc[lambda n: n >= 20].index
taux = ac.taux_lexique(df, "public").loc[grands_publics].T
taux = taux.loc[taux.mean(axis=1).sort_values(ascending=False).index]
taux.columns = [textwrap.fill(p, 16).replace("\n", "<br>") for p in taux.columns]

g.carte_chaleur(taux, "Mentions de chaque thème pour 10 000 mots, par public",
                "Chaque ligne va du plus clair (minimum) au plus foncé (maximum)").show()

> À rédiger. Le vocabulaire change-t-il plus selon le public ou selon le continent ? Quels thèmes sont réservés à un public précis ?

### Les mots propres à chaque continent

Pour chaque continent, on cherche les mots plus fréquents que dans le reste du corpus. Le score utilisé, le log-odds avec a priori de Dirichlet (Monroe, Colaresi et Quinn, 2008), tient compte de la taille de chaque groupe et limite le poids des mots rares. Au-dessus de 2, l'écart est net. Les noms de lieux ressortent logiquement en tête.

In [11]:
mots_par_continent = ac.mots_caracteristiques(df, "continent", n=10)
continents = [c for c in ac.ORDRE_CONTINENTS if c in mots_par_continent]

fig = make_subplots(rows=1, cols=len(continents), subplot_titles=continents, shared_xaxes=True,
                    horizontal_spacing=0.08)
for colonne, continent in enumerate(continents, start=1):
    scores = mots_par_continent[continent].iloc[::-1]
    fig.add_bar(x=scores.values, y=scores.index, orientation="h", row=1, col=colonne, showlegend=False,
                marker_color=g.COULEUR_CONTINENT[continent],
                hovertemplate="%{y} : score %{x:.1f}<extra></extra>")
plus_fort = max(scores.max() for scores in mots_par_continent.values())
fig.update_xaxes(range=[0, plus_fort * 1.05], showgrid=True)
g.titre(fig, "Les 10 mots les plus caractéristiques de chaque continent",
        "Score z du log-odds, au-dessus de 2 l'écart est net", hauteur=480)
fig.show()

> À rédiger. Au-delà des noms de lieux, quels mots ressortent ? Que disent-ils des sujets propres à chaque région ?

### Le discours évolue-t-il dans le temps ?

On suit quatre thèmes année par année. Le trait gris donne la moyenne sur tous les voyages. Certaines années ne comptent qu'un ou deux voyages, les variations y sont donc fragiles.

In [12]:
themes = ["Paix et guerre", "Migrants et réfugiés", "Écologie et création", "Dialogue interreligieux"]
par_annee = ac.taux_lexique(df, "annee").reindex(range(2013, 2025))
moyenne = ac.taux_lexique(df.assign(tous="ensemble"), "tous").loc["ensemble"]

fig = make_subplots(rows=2, cols=2, subplot_titles=themes, shared_xaxes=True,
                    vertical_spacing=0.16, horizontal_spacing=0.08)
for i, (theme, couleur) in enumerate(zip(themes, g.COULEURS)):
    ligne, colonne = i // 2 + 1, i % 2 + 1
    fig.add_scatter(x=par_annee.index, y=par_annee[theme], mode="lines+markers", row=ligne, col=colonne,
                    line=dict(color=couleur, width=2), marker=dict(size=8, line=dict(color=g.FOND, width=1.5)),
                    showlegend=False, hovertemplate="%{x} : %{y:.1f} mentions pour 10 000 mots<extra></extra>")
    fig.add_hline(y=moyenne[theme], line=dict(color=g.GRIS, width=1), layer="below", row=ligne, col=colonne)
fig.update_xaxes(dtick=2)
fig.update_yaxes(rangemode="tozero", showgrid=True)
g.titre(fig, "Mentions pour 10 000 mots, année par année",
        "Le trait gris donne la moyenne sur tous les voyages", hauteur=580)
fig.show()

> À rédiger. Certains thèmes prennent-ils de l'importance ou en perdent-ils ? Les pics correspondent-ils à des voyages précis ?

## 4. Limites

- Les textes analysés sont les versions anglaises publiées par le Vatican. Une grande partie des discours a été prononcée en espagnol ou en italien, on travaille donc souvent sur une traduction.
- Le pays et le public sont déduits automatiquement de l'en-tête et du titre. Quelques erreurs restent possibles, par exemple « Georgia » peut désigner l'État américain.
- Les thèmes reposent sur des listes de mots choisies par nous. Un mot peut avoir un autre sens, et une idée peut s'exprimer sans ces mots.
- Les groupes sont de tailles inégales. L'Océanie compte 5 discours et certaines années un seul voyage. Les écarts entre petits groupes sont à lire avec prudence.

> À rédiger. Complétez avec vos propres remarques sur les données et la méthode.

## 5. Conclusion

> À rédiger. La réponse à la question de départ, en vous appuyant sur les graphiques ci-dessus. Le pape adapte-t-il son discours, et sur quels points ?